In [1]:
import pandas as pd

In [2]:
stock = pd.read_excel(r"/home/VECVNET/zzasom/jupyter/01_Developement_work_QA/Chakshu_G/order_orchestration/order_orchestration_mannual/Build/data/cleaned/inventory_master_og.xlsx",engine='openpyxl')

In [3]:
stock.duplicated().sum()

0

In [4]:
stock.head()

,snapshot_timestamp,location,material_code,actual_stock,allocated_quantity,pending_so_quantity,safety_stock,ageing_quantity,ageing_quantity_gt_1yr
0,2026-01-01,1110,0031110,0,0,0,0,0,0
1,2026-01-01,1110,0031111,0,0,0,0,0,0
2,2026-01-01,1110,0031301,14,0,0,0,14,14
3,2026-01-01,1110,0031302,11,0,0,0,11,11
4,2026-01-01,1110,0031401,5,0,0,0,0,0


In [5]:
open_order = pd.read_excel(r"/home/VECVNET/zzasom/jupyter/01_Developement_work_QA/Chakshu_G/order_orchestration/order_orchestration_mannual/Input/modified_input_files/input_open_order_mapped.xlsx",engine='openpyxl')

In [6]:
open_order.columns

Index(['Customer', 'SaTy', 'Doc. Date ', 'Time    .1', 'Material   ',
       'Open Order Qty.', 'Plnt', 'EAN/UPC', 'Date      ', 'Time    ',
       'Cust Type', 'Material Description                    ',
       'Name 1                        ', 'CustomerStatus'],
      dtype='object')

In [7]:
open_order["Plnt"] = open_order["Plnt"].astype(str)

In [8]:
open_order[
    (open_order["Plnt"] == "1155") &
    (open_order["Material   "] == "IA306245")
]

,Customer,SaTy,Doc. Date,Time .1,Material,Open Order Qty.,Plnt,EAN/UPC,Date,Time,Cust Type,Material Description,Name 1,CustomerStatus


In [9]:
open_order.duplicated().sum()

4

In [10]:
open_order.head()

,Customer,SaTy,Doc. Date,Time .1,Material,Open Order Qty.,Plnt,EAN/UPC,Date,Time,Cust Type,Material Description,Name 1,CustomerStatus
0,1XS1299,ZESS,2024-11-14,11:52:31,IA236530,1,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
1,1S6265,ZEOR,2026-08-19,10:11:16,ST000213M,1,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
2,1S6228,ZMSO,2026-08-18,18:42:13,ID379455,2,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
3,1S2150,ZROH,2026-08-11,17:18:20,ST111341,1,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
4,1S3452,ZROH,2026-08-13,12:41:19,IA460255,1,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0


In [11]:
Pending_SO_orders = open_order[
    (open_order["SaTy"].isin(["ZACC", "ZROP"])) |
    (open_order["Customer"].astype(str).str.startswith("1XS")) |
    (open_order["EAN/UPC"].isin([11, 12, 13, 14, 15]))
]

In [12]:
Pending_SO_orders.head()

,Customer,SaTy,Doc. Date,Time .1,Material,Open Order Qty.,Plnt,EAN/UPC,Date,Time,Cust Type,Material Description,Name 1,CustomerStatus
0,1XS1299,ZESS,2024-11-14,11:52:31,IA236530,1,1110,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
6,1XS1182,ZESS,2026-08-13,16:39:36,IU481982,10,9045,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
7,1XS1182,ZESS,2026-08-13,16:39:36,ID319332,1,9045,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
8,1XS1182,ZESS,2026-08-13,16:39:36,IA222133,1,9045,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0
9,1XS1182,ZESS,2026-08-13,16:39:36,ID313220,1,9045,1,2026-11-13,03:02:09,DOMESTIC,ABCD,XYZ,0


In [13]:
open_order.columns

Index(['Customer', 'SaTy', 'Doc. Date ', 'Time    .1', 'Material   ',
       'Open Order Qty.', 'Plnt', 'EAN/UPC', 'Date      ', 'Time    ',
       'Cust Type', 'Material Description                    ',
       'Name 1                        ', 'CustomerStatus'],
      dtype='object')

In [14]:
alloc_qty = (
    Pending_SO_orders 
    .groupby(["Plnt", "Material   "], as_index=False)["Open Order Qty."]
    .sum()
    .rename(columns={"Open Order Qty.": "pending_so_quantity"})
)

In [15]:
alloc_qty.head()

,Plnt,Material,pending_so_quantity
0,1110,0032222,1
1,1110,2200011998,20
2,1110,2200470467,1
3,1110,2200925091,50
4,1110,2200973950,2


In [16]:
inventory_master = stock

In [17]:
inventory_master.head()

,snapshot_timestamp,location,material_code,actual_stock,allocated_quantity,pending_so_quantity,safety_stock,ageing_quantity,ageing_quantity_gt_1yr
0,2026-01-01,1110,0031110,0,0,0,0,0,0
1,2026-01-01,1110,0031111,0,0,0,0,0,0
2,2026-01-01,1110,0031301,14,0,0,0,14,14
3,2026-01-01,1110,0031302,11,0,0,0,11,11
4,2026-01-01,1110,0031401,5,0,0,0,0,0


In [18]:
inventory_master["location"] = inventory_master["location"].astype(str)
inventory_master["material_code"] = inventory_master["material_code"].astype(str)

alloc_qty["Plnt"] = alloc_qty["Plnt"].astype(str)
alloc_qty["Material   "] = alloc_qty["Material   "].astype(str)

alloc_qty = alloc_qty.rename(
    columns={"pending_so_quantity": "pending_so_quantity_new"}
)

inventory_master = inventory_master.merge(
    alloc_qty,
    left_on=["location", "material_code"],
    right_on=["Plnt", "Material   "],
    how="left"
)

inventory_master["pending_so_quantity"] = (
    inventory_master["pending_so_quantity_new"]
    .fillna(0)
)

inventory_master.drop(
    columns=[
        "Plnt",
        "Material   ",
        "pending_so_quantity_new"
    ],
    inplace=True,
    errors="ignore"
)
# inventory_master = inventory_master.merge(
#     alloc_qty,
#     left_on=["location", "material_code"],
#     right_on=["Plnt", "Material   "],
#     how="left"
# )

# inventory_master["allocated_quantity"] = (
#     inventory_master["allocated_quantity_y"]
#     .fillna(0)
# )

# inventory_master.drop(
#     columns=["Plnt", "Material   ", "allocated_quantity_y"],
#     inplace=True,
#     errors="ignore"
# )

In [19]:
inventory_master.head()

,snapshot_timestamp,location,material_code,actual_stock,allocated_quantity,pending_so_quantity,safety_stock,ageing_quantity,ageing_quantity_gt_1yr
0,2026-01-01,1110,0031110,0,0,0.0,0,0,0
1,2026-01-01,1110,0031111,0,0,0.0,0,0,0
2,2026-01-01,1110,0031301,14,0,0.0,0,14,14
3,2026-01-01,1110,0031302,11,0,0.0,0,11,11
4,2026-01-01,1110,0031401,5,0,0.0,0,0,0


In [20]:
inventory_master.shape

(296985, 9)

In [21]:
inventory_master.dtypes

snapshot_timestamp        datetime64[ns]
location                          object
material_code                     object
actual_stock                       int64
allocated_quantity                 int64
pending_so_quantity              float64
safety_stock                       int64
ageing_quantity                    int64
ageing_quantity_gt_1yr             int64
dtype: object

In [22]:
inventory_master['pending_so_quantity'].value_counts()

0.0      283743
1.0        5432
2.0        2647
3.0        1118
4.0         791
          ...  
144.0         1
511.0         1
718.0         1
99.0          1
104.0         1
Name: pending_so_quantity, Length: 129, dtype: int64

In [23]:
inventory_master.duplicated().sum()

0

In [24]:
inventory_master.to_excel(r"/home/VECVNET/zzasom/jupyter/01_Developement_work_QA/Chakshu_G/order_orchestration/order_orchestration_mannual/Build/data/cleaned/inventory_master.xlsx",index=False)

In [26]:
# import pandas as pd

# input_file = r"C:\Order Orchestration code\order_orchestration\original_files_08092026\Combained Sheet 08.09.2026.xlsx" 
# output_file = r"c:\Order Orchestration code\order_orchestration\data\cleaned\cleaned_ageing_plant_material_level.xlsx"

# xls = pd.ExcelFile(input_file)

# all_data = []

# for sheet_name in xls.sheet_names:

#     if sheet_name.strip().lower() == "inventory ageing summary":
#         continue

#     try:

#         df = pd.read_excel(
#             input_file,
#             sheet_name=sheet_name,
#             header=1,
#             engine="openpyxl"
#         )

#         # Clean column names
#         df.columns = [str(col).strip() for col in df.columns]

#         # Dynamic column identification
#         plant_col = next(
#             col for col in df.columns
#             if str(col).strip().lower() == "plant"
#         )

#         material_col = next(
#             col for col in df.columns
#             if "material" in str(col).lower()
#         )

#         stock_col = next(
#             col for col in df.columns
#             if "stock" in str(col).lower()
#         )

#         gt1_col = next(
#             col for col in df.columns
#             if "> 1" in str(col)
#         )

#         gt2_col = next(
#             col for col in df.columns
#             if "> 2" in str(col)
#         )

#         gt3_col = next(
#             col for col in df.columns
#             if "> 3" in str(col)
#         )

#         temp = pd.DataFrame()

#         temp["plant"] = df[plant_col]
#         temp["material_code"] = df[material_col]

#         temp["ageing_stock_qty"] = pd.to_numeric(
#             df[stock_col],
#             errors="coerce"
#         ).fillna(0)

#         temp["ageing_stock_gt_1yr"] = (
#             pd.to_numeric(df[gt1_col], errors="coerce").fillna(0)
#             + pd.to_numeric(df[gt2_col], errors="coerce").fillna(0)
#             + pd.to_numeric(df[gt3_col], errors="coerce").fillna(0)
#         )

#         temp["is_ageing_stock_available"] = (
#             temp["ageing_stock_gt_1yr"] > 0
#         ).astype(int)

#         all_data.append(temp)

#         print(f"Processed: {sheet_name}")

#     except Exception as e:
#         print(f"Skipped {sheet_name}: {e}")

# if not all_data:
#     raise ValueError("No valid sheets processed.")

# final_df = pd.concat(all_data, ignore_index=True)

# # Remove invalid/header rows
# final_df["plant"] = pd.to_numeric(
#     final_df["plant"],
#     errors="coerce"
# )

# # final_df["material_code"] = pd.to_numeric(
# #     final_df["material_code"],
# #     errors="coerce"
# # )

# final_df = final_df[
#     final_df["plant"].notna()
# ]

# final_df = final_df[
#     final_df["material_code"].notna()
# ]

# final_df["plant"] = final_df["plant"].astype(int)
# final_df["material_code"] = final_df["material_code"].astype(str).str.strip()

# # Aggregate duplicates
# final_df = (
#     final_df.groupby(
#         ["plant", "material_code"],
#         as_index=False
#     )
#     .agg({
#         "ageing_stock_qty": "sum",
#         "ageing_stock_gt_1yr": "sum"
#     })
# )

# # Create ageing availability flag
# final_df["is_ageing_stock_available"] = (
#     final_df["ageing_stock_gt_1yr"] > 0
# ).astype(int)

# # Final column order
# final_df = final_df[
#     [
#         "plant",
#         "material_code",
#         "ageing_stock_qty",
#         "ageing_stock_gt_1yr",
#         "is_ageing_stock_available"
#     ]
# ]

# final_df = final_df[final_df['ageing_stock_qty']> 0]

# # Save file
# final_df.to_excel(output_file, index=False)

# print(f"Saved: {output_file}")
# print(f"Rows created: {len(final_df):,}")